# Energy Harvesting Scheduler Simulation

This notebook converts the provided Python simulation into a structured, publication-ready Colab notebook based on the uploaded source file. It simulates an energy-harvesting communication system with packet generation, finite battery storage, and multiple scheduling policies. The notebook evaluates how different transmission strategies behave under varying energy arrival rates.

The workflow is organized so that:
- each code cell is preceded by a short explanatory text cell,
- all generated figures and tables are saved to Google Drive,
- results are reproducible through a fixed random seed,
- the final outputs include CSV tables, PNG figures, and a text summary for convenient reporting.

**Source transformed into notebook:** uploaded file `energy_harvesting_simulator.py`. 

## Environment setup and Google Drive output folders

This cell imports the required libraries, mounts Google Drive when running in Colab, and creates a dedicated output directory where all generated tables, figures, and summaries will be stored.

In [ ]:

from pathlib import Path
import os
import random
import simpy
import pandas as pd
import matplotlib.pyplot as plt

# --------------------------------------------
# Optional Google Drive mounting (for Colab)
# --------------------------------------------
IN_COLAB = False
try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except ImportError:
    drive = None

if IN_COLAB:
    drive.mount('/content/drive')

# --------------------------------------------
# Configure output directories
# --------------------------------------------
if IN_COLAB:
    BASE_OUTPUT_DIR = Path("/content/drive/MyDrive/Outputs/EnergyHarvestingScheduler")
else:
    BASE_OUTPUT_DIR = Path.cwd() / "Outputs" / "EnergyHarvestingScheduler"

FIG_DIR = BASE_OUTPUT_DIR / "figures"
TABLE_DIR = BASE_OUTPUT_DIR / "tables"
TEXT_DIR = BASE_OUTPUT_DIR / "others"

for folder in [FIG_DIR, TABLE_DIR, TEXT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Base output directory:", BASE_OUTPUT_DIR)
print("Figures:", FIG_DIR)
print("Tables:", TABLE_DIR)
print("Other outputs:", TEXT_DIR)


## Simulation constants and reproducibility controls

This cell defines the simulation time unit and fixes the random seed so the experiments remain reproducible across runs.

In [ ]:

TIME_UNIT = "simulation time units"
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

print(f"Random seed fixed at: {RANDOM_SEED}")
print(f"Time unit: {TIME_UNIT}")


## Energy harvester model

This cell defines the energy harvesting process. Energy arrives according to an exponential inter-arrival model and is stored in the battery whenever capacity is available.

In [ ]:

class EnergyHarvester:
    def __init__(self, env, battery, arrival_rate, energy_unit):
        self.env = env
        self.battery = battery
        self.arrival_rate = arrival_rate
        self.energy_unit = energy_unit
        self.env.process(self.harvest())

    def harvest(self):
        """Simulate stochastic energy arrivals."""
        while True:
            yield self.env.timeout(random.expovariate(self.arrival_rate))
            if self.battery.level < self.battery.capacity:
                yield self.battery.put(self.energy_unit)


## Packet generation model

This cell defines the packet arrival process. Packets are generated using an exponential inter-arrival distribution and stored in a transmission buffer together with their generation timestamps.

In [ ]:

class PacketGenerator:
    def __init__(self, env, arrival_rate, buffer):
        self.env = env
        self.arrival_rate = arrival_rate
        self.buffer = buffer
        self.packets_generated = 0
        self.env.process(self.generate())

    def generate(self):
        """Simulate stochastic packet arrivals."""
        while True:
            yield self.env.timeout(random.expovariate(self.arrival_rate))
            self.packets_generated += 1
            yield self.buffer.put(self.env.now)


## Scheduling policies and transmission logic

This cell implements the scheduler. It supports multiple policies (`greedy`, `threshold`, `random`, and `save_then_transmit`) and models transmission success over an abstracted wireless link.

In [ ]:

class Scheduler:
    def __init__(self, env, battery, buffer, policy,
                 energy_per_packet, threshold, random_p,
                 link_success_prob=0.95):
        self.env = env
        self.battery = battery
        self.buffer = buffer
        self.policy = policy
        self.energy_per_packet = energy_per_packet
        self.threshold = threshold
        self.random_p = random_p
        self.link_success_prob = link_success_prob

        self.sent_packets = 0
        self.lost_packets = 0
        self.total_delay = 0

        self.env.process(self.run())

    def run(self):
        """Main scheduling loop."""
        while True:
            if len(self.buffer.items) == 0:
                yield self.env.timeout(1)
                continue

            if self.policy == "greedy":
                decision = True
            elif self.policy == "threshold":
                decision = self.battery.level >= self.threshold
            elif self.policy == "random":
                decision = random.random() < self.random_p
            elif self.policy == "save_then_transmit":
                decision = self.battery.level >= self.threshold
            else:
                decision = True

            if decision and self.battery.level >= self.energy_per_packet:
                pkt_time = yield self.buffer.get()
                yield self.battery.get(self.energy_per_packet)

                if random.random() < self.link_success_prob:
                    self.sent_packets += 1
                    self.total_delay += (self.env.now - pkt_time)
                else:
                    self.lost_packets += 1

                yield self.env.timeout(1)
            else:
                yield self.env.timeout(1)


## Single-scenario simulation runner

This cell wraps one complete simulation experiment for a selected policy and energy arrival rate, then computes the main performance indicators.

In [ ]:

def run_simulation(policy, energy_rate, sim_time=500):
    env = simpy.Environment()
    battery = simpy.Container(env, capacity=20, init=5)
    buffer = simpy.Store(env)

    EnergyHarvester(env, battery, arrival_rate=energy_rate, energy_unit=2)
    traffic = PacketGenerator(env, arrival_rate=1.0, buffer=buffer)

    scheduler = Scheduler(
        env, battery, buffer, policy,
        energy_per_packet=3,
        threshold=18,
        random_p=0.5,
        link_success_prob=0.95
    )

    env.run(until=sim_time)

    total_generated = traffic.packets_generated
    total_sent = scheduler.sent_packets
    total_lost = total_generated - total_sent
    avg_delay = scheduler.total_delay / total_sent if total_sent > 0 else 0
    delivery_ratio = total_sent / total_generated if total_generated > 0 else 0

    return {
        "Policy": policy,
        "Energy_Arrival_Rate": energy_rate,
        "Generated": total_generated,
        "Sent": total_sent,
        "Lost": total_lost,
        f"Avg_Delay ({TIME_UNIT})": avg_delay,
        "Delivery_Ratio": delivery_ratio
    }


## Experimental design

This cell defines the candidate scheduling policies and energy arrival rates used in the comparative study.

In [ ]:

policies = ["greedy", "threshold", "random", "save_then_transmit"]
energy_rates = [0.5, 0.8, 1.0]

print("Policies:", policies)
print("Energy arrival rates:", energy_rates)


## Execute all simulations and build the results table

This cell runs the full experiment matrix and stores the aggregated results in a pandas DataFrame.

In [ ]:

results = []

# Reset seed for reproducibility before the full run
random.seed(RANDOM_SEED)

for e_rate in energy_rates:
    for policy in policies:
        results.append(run_simulation(policy, e_rate))

df = pd.DataFrame(results)

print("\n=== Simulation Results ===")
print("Note: Time-related metrics are expressed in simulation time units.\n")
display(df)


## Save result tables to Google Drive

This cell saves the main results table in CSV format and also exports a rounded copy for easier reporting.

In [ ]:

results_csv_path = TABLE_DIR / "simulation_results.csv"
results_report_path = TABLE_DIR / "simulation_results_rounded.csv"

df.to_csv(results_csv_path, index=False)
df.round(4).to_csv(results_report_path, index=False)

print("Saved main results table to:", results_csv_path)
print("Saved rounded reporting table to:", results_report_path)


## Generate and save bar charts for each energy rate

This cell creates separate bar charts for packets sent, packets lost, and average delay under each energy arrival rate, and saves them as PNG files to Google Drive.

In [ ]:

colors = {
    "greedy": "green",
    "threshold": "orange",
    "random": "red",
    "save_then_transmit": "purple"
}

saved_bar_figures = []

for e_rate in energy_rates:
    subset = df[df["Energy_Arrival_Rate"] == e_rate]

    # Packets Sent
    fig = plt.figure(figsize=(7, 4))
    plt.bar(subset["Policy"], subset["Sent"], color=[colors[p] for p in subset["Policy"]])
    plt.ylabel("Packets Sent")
    plt.title(f"Packets Sent vs Policy (Energy Rate = {e_rate})")
    plt.xticks(rotation=15)
    plt.tight_layout()
    out_path = FIG_DIR / f"bar_packets_sent_rate_{str(e_rate).replace('.', '_')}.png"
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    saved_bar_figures.append(out_path)
    plt.show()

    # Packets Lost
    fig = plt.figure(figsize=(7, 4))
    plt.bar(subset["Policy"], subset["Lost"], color=[colors[p] for p in subset["Policy"]])
    plt.ylabel("Packets Lost")
    plt.title(f"Packets Lost vs Policy (Energy Rate = {e_rate})")
    plt.xticks(rotation=15)
    plt.tight_layout()
    out_path = FIG_DIR / f"bar_packets_lost_rate_{str(e_rate).replace('.', '_')}.png"
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    saved_bar_figures.append(out_path)
    plt.show()

    # Average Delay
    fig = plt.figure(figsize=(7, 4))
    plt.bar(
        subset["Policy"],
        subset[f"Avg_Delay ({TIME_UNIT})"],
        color=[colors[p] for p in subset["Policy"]]
    )
    plt.ylabel(f"Average Delay ({TIME_UNIT})")
    plt.title(f"Average Delay vs Policy (Energy Rate = {e_rate})")
    plt.xticks(rotation=15)
    plt.tight_layout()
    out_path = FIG_DIR / f"bar_avg_delay_rate_{str(e_rate).replace('.', '_')}.png"
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    saved_bar_figures.append(out_path)
    plt.show()

print("Saved bar-chart figures:")
for path in saved_bar_figures:
    print(" -", path)


## Generate and save cross-rate line charts

This cell creates policy-wise line charts to compare throughput, average delay, and delivery ratio across different energy arrival rates.

In [ ]:

saved_line_figures = []

# Throughput vs Energy Budget
plt.figure(figsize=(7, 4))
for policy in policies:
    subset = df[df["Policy"] == policy]
    plt.plot(subset["Energy_Arrival_Rate"], subset["Sent"],
             marker='o', label=policy, color=colors[policy])
plt.xlabel("Energy Arrival Rate")
plt.ylabel("Packets Sent")
plt.title("Throughput vs Energy Budget")
plt.legend()
plt.grid(True)
plt.tight_layout()
out_path = FIG_DIR / "line_throughput_vs_energy_budget.png"
plt.savefig(out_path, dpi=300, bbox_inches="tight")
saved_line_figures.append(out_path)
plt.show()

# Average Delay vs Energy Budget
plt.figure(figsize=(7, 4))
for policy in policies:
    subset = df[df["Policy"] == policy]
    plt.plot(subset["Energy_Arrival_Rate"],
             subset[f"Avg_Delay ({TIME_UNIT})"],
             marker='o', label=policy, color=colors[policy])
plt.xlabel("Energy Arrival Rate")
plt.ylabel(f"Average Delay ({TIME_UNIT})")
plt.title("Average Delay vs Energy Budget")
plt.legend()
plt.grid(True)
plt.tight_layout()
out_path = FIG_DIR / "line_avg_delay_vs_energy_budget.png"
plt.savefig(out_path, dpi=300, bbox_inches="tight")
saved_line_figures.append(out_path)
plt.show()

# Delivery Ratio vs Energy Budget
plt.figure(figsize=(7, 4))
for policy in policies:
    subset = df[df["Policy"] == policy]
    plt.plot(subset["Energy_Arrival_Rate"], subset["Delivery_Ratio"],
             marker='o', label=policy, color=colors[policy])
plt.xlabel("Energy Arrival Rate")
plt.ylabel("Packet Delivery Ratio")
plt.title("Delivery Ratio vs Energy Budget")
plt.legend()
plt.grid(True)
plt.tight_layout()
out_path = FIG_DIR / "line_delivery_ratio_vs_energy_budget.png"
plt.savefig(out_path, dpi=300, bbox_inches="tight")
saved_line_figures.append(out_path)
plt.show()

print("Saved line-chart figures:")
for path in saved_line_figures:
    print(" -", path)


## Export a concise text summary of the experiment

This cell writes a compact text summary to Google Drive so the key outcomes can be easily reused in reports or manuscripts.

In [ ]:

summary_lines = []
summary_lines.append("Energy Harvesting Scheduler Simulation - Outputs Summary")
summary_lines.append("=" * 60)
summary_lines.append(f"Random seed: {RANDOM_SEED}")
summary_lines.append(f"Time unit: {TIME_UNIT}")
summary_lines.append("")
summary_lines.append("Policies evaluated: " + ", ".join(policies))
summary_lines.append("Energy arrival rates: " + ", ".join(map(str, energy_rates)))
summary_lines.append("")
summary_lines.append("Main results table:")
summary_lines.append(df.round(4).to_string(index=False))
summary_lines.append("")
summary_lines.append("Best policy by delivery ratio within each energy arrival rate:")

best_by_rate = (
    df.sort_values(["Energy_Arrival_Rate", "Delivery_Ratio"], ascending=[True, False])
      .groupby("Energy_Arrival_Rate", as_index=False)
      .first()
)

for _, row in best_by_rate.iterrows():
    summary_lines.append(
        f"- Energy rate {row['Energy_Arrival_Rate']}: "
        f"{row['Policy']} "
        f"(Delivery Ratio={row['Delivery_Ratio']:.4f}, "
        f"Sent={int(row['Sent'])}, "
        f"Avg Delay={row[f'Avg_Delay ({TIME_UNIT})']:.4f})"
    )

summary_path = TEXT_DIR / "outputs_summary.txt"
with open(summary_path, "w", encoding="utf-8") as f:
    f.write("\n".join(summary_lines))

print("Saved outputs summary to:", summary_path)
print("\n".join(summary_lines[:12]))


## Final note

After execution in Google Colab, all outputs will be stored under the selected Google Drive folder:
`MyDrive/Outputs/EnergyHarvestingScheduler/`

This includes:
- `figures/` for PNG charts,
- `tables/` for CSV result tables,
- `others/` for the text summary file.